In [1]:
import sys

import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

d:\Zfs_Clip_Image\zfs-clip-image-captioning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.13.0+cpu
Transformers: 5.15.0


In [2]:
import json
from pathlib import Path

import numpy as np
import torch

from transformers import AutoTokenizer

In [3]:
from config import (
    PROJECT_ROOT,
    DATA_ROOT,
    SPLIT_DIR,
    SUBSET_DIR,
    TOKENIZED_DIR,
)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("SPLIT_DIR    :", SPLIT_DIR)
print("SUBSET_DIR   :", SUBSET_DIR)
print("TOKENIZED_DIR:", TOKENIZED_DIR)

PROJECT_ROOT : d:\Zfs_Clip_Image\zfs-clip-image-captioning
DATA_ROOT    : d:\Zfs_Clip_Image\zfs-clip-image-captioning\notebook\data\flickr8k
SPLIT_DIR    : d:\Zfs_Clip_Image\zfs-clip-image-captioning\notebook\data\flickr8k\splits
SUBSET_DIR   : d:\Zfs_Clip_Image\zfs-clip-image-captioning\notebook\data\flickr8k\subsets
TOKENIZED_DIR: d:\Zfs_Clip_Image\zfs-clip-image-captioning\notebook\data\flickr8k\tokenized


In [4]:
# Đường dẫn tới các frozen split từ bước 1-6
split_paths = {
    "train": SPLIT_DIR / "train.json",
    "val": SPLIT_DIR / "val.json",
    "test": SPLIT_DIR / "test.json",
}

for split_name, split_path in split_paths.items():
    print(
        f"{split_name:5} | "
        f"exists = {split_path.exists()} | "
        f"path = {split_path}"
    )

train | exists = True | path = d:\Zfs_Clip_Image\zfs-clip-image-captioning\notebook\data\flickr8k\splits\train.json
val   | exists = True | path = d:\Zfs_Clip_Image\zfs-clip-image-captioning\notebook\data\flickr8k\splits\val.json
test  | exists = True | path = d:\Zfs_Clip_Image\zfs-clip-image-captioning\notebook\data\flickr8k\splits\test.json


In [5]:
#Load dữ các file test train val 
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


train_data = load_json(split_paths["train"])
val_data = load_json(split_paths["val"])
test_data = load_json(split_paths["test"])

print("Train type:", type(train_data))
print("Val type  :", type(val_data))
print("Test type :", type(test_data))

print()

print("Train size:", len(train_data))
print("Val size  :", len(val_data))
print("Test size :", len(test_data))

Train type: <class 'dict'>
Val type  : <class 'dict'>
Test type : <class 'dict'>

Train size: 6464
Val size  : 808
Test size : 809


In [6]:
#Kiểm tra 1 sample thật trong tập dữ liệu
if isinstance(train_data, dict):
    first_image_id = next(iter(train_data))
    
    print("First image_id:")
    print(first_image_id)
    
    print("\nValue:")
    print(train_data[first_image_id])

elif isinstance(train_data, list):
    print("First item:")
    print(train_data[0])

else:
    print("Unexpected data type:", type(train_data))

First image_id:
3472540184_b0420b921a.jpg

Value:
['An older boy chases a laughing younger boy on the grass .', "Two boys are running ; one 's smiling and being touched by the other .", 'Two children run and play in the grass .', 'Two young boys are running through a grassy area .', 'Two young boys run across a green yard .']


In [7]:
def validate_split(split_name, data):
    # Kiểm tra toàn bộ split có đúng kiểu dictionary không
    assert isinstance(data, dict), \
        f"{split_name} must be a dictionary."

    # Những image có captions không phải list
    invalid_caption_lists = [
        image_id
        for image_id, captions in data.items()
        if not isinstance(captions, list)
    ]

    # Số caption của từng image
    caption_counts = [
        len(captions)
        for captions in data.values()
        if isinstance(captions, list)
    ]

    # Caption nào không phải string
    invalid_caption_types = [
        (image_id, caption)
        for image_id, captions in data.items()
        if isinstance(captions, list)
        for caption in captions
        if not isinstance(caption, str)
    ]

    # Caption rỗng hoặc chỉ toàn khoảng trắng
    empty_captions = [
        (image_id, caption)
        for image_id, captions in data.items()
        if isinstance(captions, list)
        for caption in captions
        if isinstance(caption, str) and not caption.strip()
    ]

    total_captions = sum(caption_counts)

    print("Images               :", len(data))
    print("Total captions       :", total_captions)
    print("Caption count/image  :", sorted(set(caption_counts)))
    print("Invalid caption lists:", len(invalid_caption_lists))
    print("Invalid caption types:", len(invalid_caption_types))
    print("Empty captions       :", len(empty_captions))
    print()

    return {
        "invalid_caption_lists": invalid_caption_lists,
        "invalid_caption_types": invalid_caption_types,
        "empty_captions": empty_captions,
    }


train_check = validate_split("train", train_data)
val_check = validate_split("val", val_data)
test_check = validate_split("test", test_data)

Images               : 6464
Total captions       : 32320
Caption count/image  : [5]
Invalid caption lists: 0
Invalid caption types: 0
Empty captions       : 0

Images               : 808
Total captions       : 4040
Caption count/image  : [5]
Invalid caption lists: 0
Invalid caption types: 0
Empty captions       : 0

Images               : 809
Total captions       : 4045
Caption count/image  : [5]
Invalid caption lists: 0
Invalid caption types: 0
Empty captions       : 0



In [8]:
def flatten_split(data):
    samples = []

    for image_id, captions in data.items():
        for caption_idx, caption in enumerate(captions):
            samples.append({
                "image_id": image_id,
                "caption_idx": caption_idx,
                "caption": caption,
            })

    return samples


train_samples = flatten_split(train_data)
val_samples = flatten_split(val_data)
test_samples = flatten_split(test_data)

print("Train samples:", len(train_samples))
print("Val samples  :", len(val_samples))
print("Test samples :", len(test_samples))

Train samples: 32320
Val samples  : 4040
Test samples : 4045


In [9]:
from config import GPT2_MODEL_NAME
print("GPT2_MODEL   :", GPT2_MODEL_NAME)

GPT2_MODEL   : openai-community/gpt2


In [10]:
# load tokenizer từ GPT2
tokenizer = AutoTokenizer.from_pretrained(
    GPT2_MODEL_NAME,
    use_fast=True,
)

print("Tokenizer class :", tokenizer.__class__.__name__)
print("Vocabulary size :", len(tokenizer))
print("Model max length:", tokenizer.model_max_length)

print("BOS token       :", tokenizer.bos_token)
print("EOS token       :", tokenizer.eos_token)
print("PAD token       :", tokenizer.pad_token)

Tokenizer class : GPT2Tokenizer
Vocabulary size : 50257
Model max length: 1024
BOS token       : <|endoftext|>
EOS token       : <|endoftext|>
PAD token       : None


In [11]:
# Thử tách token trên 1 caption
sample = train_samples[0]
sample_caption = sample["caption"]

print("Image ID   :", sample["image_id"])
print("Caption idx:", sample["caption_idx"])
print("Caption    :", sample_caption)

Image ID   : 3472540184_b0420b921a.jpg
Caption idx: 0
Caption    : An older boy chases a laughing younger boy on the grass .


In [12]:
# Dùng tokenize cho 1 caption vừa test
encoded = tokenizer(
    sample_caption,
    add_special_tokens=False,
)

print(encoded)

{'input_ids': [2025, 4697, 2933, 442, 1386, 257, 14376, 7099, 2933, 319, 262, 8701, 764], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [13]:
# Check kỹ bên trong 1 tokenize
input_ids = encoded["input_ids"]

tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Original caption:")
print(sample_caption)

print("\nTokens:")
print(tokens)

print("\nInput IDs:")
print(input_ids)

print("\nAttention mask:")
print(encoded["attention_mask"])

print("\nDecoded:")
print(tokenizer.decode(input_ids))

Original caption:
An older boy chases a laughing younger boy on the grass .

Tokens:
['An', 'Ġolder', 'Ġboy', 'Ġch', 'ases', 'Ġa', 'Ġlaughing', 'Ġyounger', 'Ġboy', 'Ġon', 'Ġthe', 'Ġgrass', 'Ġ.']

Input IDs:
[2025, 4697, 2933, 442, 1386, 257, 14376, 7099, 2933, 319, 262, 8701, 764]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Decoded:
An older boy chases a laughing younger boy on the grass .


In [14]:
# đo độ dài token của toàn bộ train_samples
train_token_lengths = []

for sample in train_samples:
    encoded = tokenizer(
        sample["caption"],
        add_special_tokens=False,
        padding=False,
        truncation=False,
    )

    train_token_lengths.append(
        len(encoded["input_ids"])
    )

train_token_lengths = np.array(train_token_lengths)

print("Number of captions:", len(train_token_lengths))
print("Min length        :", train_token_lengths.min())
print("Mean length       :", train_token_lengths.mean())
print("Median length     :", np.median(train_token_lengths))
print("P95 length        :", np.percentile(train_token_lengths, 95))
print("P99 length        :", np.percentile(train_token_lengths, 99))
print("Max length        :", train_token_lengths.max())

Number of captions: 32320
Min length        : 1
Mean length       : 12.371503712871288
Median length     : 12.0
P95 length        : 20.0
P99 length        : 24.0
Max length        : 41


In [15]:
# Kiểm tra có bao nhiêu caption dài hơn 24
candidate_lengths = [24, 32, 40, 48, 64]

for max_length in candidate_lengths:
    num_over = np.sum(train_token_lengths > max_length)
    percentage = num_over / len(train_token_lengths) * 100

    print(
        f"MAX_LENGTH = {max_length:2} | "
        f"truncated = {num_over:4} captions "
        f"({percentage:.4f}%)"
    )

MAX_LENGTH = 24 | truncated =  279 captions (0.8632%)
MAX_LENGTH = 32 | truncated =   17 captions (0.0526%)
MAX_LENGTH = 40 | truncated =    1 captions (0.0031%)
MAX_LENGTH = 48 | truncated =    0 captions (0.0000%)
MAX_LENGTH = 64 | truncated =    0 captions (0.0000%)


In [16]:
long_indices = np.where(train_token_lengths > 32)[0]

print("Captions longer than 32 tokens:", len(long_indices))
print()

for idx in long_indices:
    sample = train_samples[idx]

    print("Image ID :", sample["image_id"])
    print("Caption  :", sample["caption"])
    print("Tokens   :", train_token_lengths[idx])
    print("-" * 80)

Captions longer than 32 tokens: 17

Image ID : 1472249944_d887c3aeda.jpg
Caption  : A woman in an orange coat and jeans is squatting on a rock wall while a blonde woman in a red jacket stands next to her on the wall checking her electronic device
Tokens   : 34
--------------------------------------------------------------------------------
Image ID : 641893280_36fd6e886a.jpg
Caption  : Two brown and white dogs , one a boxer and the other a terrier , play on a rock covered hill with a blue sky and powdery clouds in the background .
Tokens   : 34
--------------------------------------------------------------------------------
Image ID : 3280672302_2967177653.jpg
Caption  : A person wearing a black shirt is getting a piggy-back ride from a person wearing a black t-shirt walkng on a paved path in the woods .
Tokens   : 33
--------------------------------------------------------------------------------
Image ID : 1499495021_d295ce577c.jpg
Caption  : A dark haired woman wearing a brown jacke

In [17]:
short_indices = np.where(train_token_lengths == 1)[0]

print("Captions with only 1 token:", len(short_indices))
print()

for idx in short_indices:
    sample = train_samples[idx]

    print("Image ID:", sample["image_id"])
    print("Caption :", repr(sample["caption"]))
    print("-" * 80)

Captions with only 1 token: 1

Image ID: 2428275562_4bde2bc5ea.jpg
Caption : 'A'
--------------------------------------------------------------------------------


In [18]:
image_id = "2428275562_4bde2bc5ea.jpg"

for idx, caption in enumerate(train_data[image_id]):
    print(f"{idx}: {repr(caption)}")

0: 'A'
1: 'A black and white dog is climbing down a hill .'
2: 'A black and white dog is sliding down a sandy hill .'
3: 'A black and white dog wearing a red collar is digging in the dirt .'
4: 'A black dog running on a sand mound .'


In [19]:
# GPT-2 không có PAD token mặc định
# Dùng EOS token làm PAD token

tokenizer.pad_token = tokenizer.eos_token

print("EOS token    :", tokenizer.eos_token)
print("EOS token ID :", tokenizer.eos_token_id)

print("PAD token    :", tokenizer.pad_token)
print("PAD token ID :", tokenizer.pad_token_id)

EOS token    : <|endoftext|>
EOS token ID : 50256
PAD token    : <|endoftext|>
PAD token ID : 50256


In [20]:
# Chốt độ dài token max 48
from config import GPT2_MAX_LENGTH

In [22]:
encoded_padded = tokenizer(
    sample_caption,
    add_special_tokens=False,
    padding="max_length",
    truncation=True,
    max_length=GPT2_MAX_LENGTH,
    return_attention_mask=True,
)

print("Input length :", len(encoded_padded["input_ids"]))
print("Mask length  :", len(encoded_padded["attention_mask"]))

print("\nInput IDs:")
print(encoded_padded["input_ids"])

print("\nAttention mask:")
print(encoded_padded["attention_mask"])

print("Real tokens   :", sum(encoded_padded["attention_mask"]))
print("Padding tokens:", GPT2_MAX_LENGTH - sum(encoded_padded["attention_mask"]))

Input length : 48
Mask length  : 48

Input IDs:
[2025, 4697, 2933, 442, 1386, 257, 14376, 7099, 2933, 319, 262, 8701, 764, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Real tokens   : 13
Padding tokens: 35


In [ ]:
# Tokenize toàn bộ tập train/val/test
def tokenize_samples(samples, tokenizer, max_length):
    captions = [sample["caption"] for sample in samples]
    
    encoded = tokenizer(
        captions,
        add_special_tokens=False,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_attention_mask=True,
        return_tensors="pt",
    )

    return encoded

In [24]:
# Tokenize trên tập train
train_encoded = tokenize_samples(
    train_samples,
    tokenizer,
    GPT2_MAX_LENGTH,
)

In [25]:
# Tokenize trên tập val
val_encoded = tokenize_samples(
    val_samples,
    tokenizer,
    GPT2_MAX_LENGTH,
)

In [27]:
# Tokenize trên tập test
test_encoded = tokenize_samples(
    test_samples,
    tokenizer,
    GPT2_MAX_LENGTH,
)

In [28]:
# Check var sau khi tokenize
print("Train input_ids      :", train_encoded["input_ids"].shape)
print("Train attention_mask :", train_encoded["attention_mask"].shape)

print("Val input_ids        :", val_encoded["input_ids"].shape)
print("Val attention_mask   :", val_encoded["attention_mask"].shape)

print("Test input_ids       :", test_encoded["input_ids"].shape)
print("Test attention_mask  :", test_encoded["attention_mask"].shape)

Train input_ids      : torch.Size([32320, 48])
Train attention_mask : torch.Size([32320, 48])
Val input_ids        : torch.Size([4040, 48])
Val attention_mask   : torch.Size([4040, 48])
Test input_ids       : torch.Size([4045, 48])
Test attention_mask  : torch.Size([4045, 48])


In [29]:
def get_token_lengths(samples, tokenizer):
    captions = [sample["caption"] for sample in samples]

    encoded = tokenizer(
        captions,
        add_special_tokens=False,
        padding=False,
        truncation=False,
    )

    return np.array([
        len(input_ids)
        for input_ids in encoded["input_ids"]
    ])


val_token_lengths = get_token_lengths(val_samples, tokenizer)
test_token_lengths = get_token_lengths(test_samples, tokenizer)

print("VAL")
print("Max length :", val_token_lengths.max())
print(
    "Over limit :",
    np.sum(val_token_lengths > GPT2_MAX_LENGTH)
)

print("\nTEST")
print("Max length :", test_token_lengths.max())
print(
    "Over limit :",
    np.sum(test_token_lengths > GPT2_MAX_LENGTH)
)

VAL
Max length : 35
Over limit : 0

TEST
Max length : 31
Over limit : 0


In [30]:
def check_encoded(split_name, samples, encoded):
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    print(f"===== {split_name.upper()} =====")

    print("Samples          :", len(samples))
    print("input_ids shape  :", input_ids.shape)
    print("mask shape       :", attention_mask.shape)

    print(
        "Mask values      :",
        torch.unique(attention_mask).tolist()
    )

    print(
        "Min real tokens  :",
        attention_mask.sum(dim=1).min().item()
    )

    print(
        "Max real tokens  :",
        attention_mask.sum(dim=1).max().item()
    )

    print()


check_encoded("train", train_samples, train_encoded)
check_encoded("val", val_samples, val_encoded)
check_encoded("test", test_samples, test_encoded)

===== TRAIN =====
Samples          : 32320
input_ids shape  : torch.Size([32320, 48])
mask shape       : torch.Size([32320, 48])
Mask values      : [0, 1]
Min real tokens  : 1
Max real tokens  : 41

===== VAL =====
Samples          : 4040
input_ids shape  : torch.Size([4040, 48])
mask shape       : torch.Size([4040, 48])
Mask values      : [0, 1]
Min real tokens  : 1
Max real tokens  : 35

===== TEST =====
Samples          : 4045
input_ids shape  : torch.Size([4045, 48])
mask shape       : torch.Size([4045, 48])
Mask values      : [0, 1]
Min real tokens  : 3
Max real tokens  : 31



In [32]:
val_short_indices = np.where(val_token_lengths == 1)[0]

print("VAL captions with only 1 token:", len(val_short_indices))
print()

for idx in val_short_indices:
    sample = val_samples[idx]

    print("Image ID:", sample["image_id"])
    print("Caption :", repr(sample["caption"]))
    print("-" * 80)

VAL captions with only 1 token: 1

Image ID: 3640443200_b8066f37f6.jpg
Caption : 'a'
--------------------------------------------------------------------------------


In [33]:
def inspect_short_captions(split_name, samples, token_lengths, max_tokens=2):
    indices = np.where(token_lengths <= max_tokens)[0]

    print(
        f"{split_name.upper()} captions "
        f"with <= {max_tokens} tokens: {len(indices)}"
    )
    print()

    for idx in indices:
        print("Image ID:", samples[idx]["image_id"])
        print("Caption :", repr(samples[idx]["caption"]))
        print("Tokens  :", token_lengths[idx])
        print("-" * 80)


inspect_short_captions(
    "train",
    train_samples,
    train_token_lengths,
)

inspect_short_captions(
    "val",
    val_samples,
    val_token_lengths,
)

inspect_short_captions(
    "test",
    test_samples,
    test_token_lengths,
)

TRAIN captions with <= 2 tokens: 9

Image ID: 3154693053_cfcd05c226.jpg
Caption : 'A basketball'
Tokens  : 2
--------------------------------------------------------------------------------
Image ID: 2165461920_1a4144eb2b.jpg
Caption : 'dogs racing'
Tokens  : 2
--------------------------------------------------------------------------------
Image ID: 2428275562_4bde2bc5ea.jpg
Caption : 'A'
Tokens  : 1
--------------------------------------------------------------------------------
Image ID: 2714703706_d21c5cb8df.jpg
Caption : 'dogs playing'
Tokens  : 2
--------------------------------------------------------------------------------
Image ID: 2929669711_b2d5a640f0.jpg
Caption : 'man surfing'
Tokens  : 2
--------------------------------------------------------------------------------
Image ID: 3189251454_03b76c2e92.jpg
Caption : 'dog barking'
Tokens  : 2
--------------------------------------------------------------------------------
Image ID: 3694071771_ce760db4c7.jpg
Caption : 'a cycli

In [34]:
def inspect_tokenized_sample(samples, encoded, idx):
    sample = samples[idx]

    input_ids = encoded["input_ids"][idx]
    attention_mask = encoded["attention_mask"][idx]

    # Chỉ lấy token thật, bỏ padding
    real_input_ids = input_ids[attention_mask.bool()]

    decoded_caption = tokenizer.decode(
        real_input_ids.tolist()
    )

    print("Image ID    :", sample["image_id"])
    print("Caption idx :", sample["caption_idx"])

    print("\nOriginal:")
    print(sample["caption"])

    print("\nDecoded:")
    print(decoded_caption)

    print("\nReal tokens:")
    print(attention_mask.sum().item())

    print("\nExact match:")
    print(decoded_caption == sample["caption"])

In [35]:
inspect_tokenized_sample(
    train_samples,
    train_encoded,
    0,
)

Image ID    : 3472540184_b0420b921a.jpg
Caption idx : 0

Original:
An older boy chases a laughing younger boy on the grass .

Decoded:
An older boy chases a laughing younger boy on the grass .

Real tokens:
13

Exact match:
True


In [36]:
inspect_tokenized_sample(
    train_samples,
    train_encoded,
    len(train_samples) // 2,
)

inspect_tokenized_sample(
    train_samples,
    train_encoded,
    len(train_samples) - 1,
)

Image ID    : 3400082864_9c737c1450.jpg
Caption idx : 0

Original:
A man in a blue headband and white shirt plays tennis .

Decoded:
A man in a blue headband and white shirt plays tennis .

Real tokens:
13

Exact match:
True
Image ID    : 2766765386_4c0beb939d.jpg
Caption idx : 4

Original:
a surfer surfs a huge wave .

Decoded:
a surfer surfs a huge wave .

Real tokens:
9

Exact match:
True


In [37]:
def build_tokenized_data(samples, encoded):
    return {
        "image_ids": [
            sample["image_id"]
            for sample in samples
        ],

        "caption_indices": [
            sample["caption_idx"]
            for sample in samples
        ],

        "captions": [
            sample["caption"]
            for sample in samples
        ],

        "input_ids": encoded["input_ids"],

        "attention_mask": encoded["attention_mask"],
    }

In [38]:
train_tokenized = build_tokenized_data(
    train_samples,
    train_encoded,
)

val_tokenized = build_tokenized_data(
    val_samples,
    val_encoded,
)

test_tokenized = build_tokenized_data(
    test_samples,
    test_encoded,
)

In [39]:
def check_tokenized_data(split_name, data):
    num_samples = len(data["image_ids"])

    print(f"===== {split_name.upper()} =====")

    print("image_ids       :", len(data["image_ids"]))
    print("caption_indices :", len(data["caption_indices"]))
    print("captions        :", len(data["captions"]))
    print("input_ids       :", data["input_ids"].shape)
    print("attention_mask  :", data["attention_mask"].shape)

    assert len(data["caption_indices"]) == num_samples
    assert len(data["captions"]) == num_samples
    assert data["input_ids"].shape[0] == num_samples
    assert data["attention_mask"].shape[0] == num_samples

    print("Alignment check : PASS")
    print()


check_tokenized_data(
    "train",
    train_tokenized,
)

check_tokenized_data(
    "val",
    val_tokenized,
)

check_tokenized_data(
    "test",
    test_tokenized,
)

===== TRAIN =====
image_ids       : 32320
caption_indices : 32320
captions        : 32320
input_ids       : torch.Size([32320, 48])
attention_mask  : torch.Size([32320, 48])
Alignment check : PASS

===== VAL =====
image_ids       : 4040
caption_indices : 4040
captions        : 4040
input_ids       : torch.Size([4040, 48])
attention_mask  : torch.Size([4040, 48])
Alignment check : PASS

===== TEST =====
image_ids       : 4045
caption_indices : 4045
captions        : 4045
input_ids       : torch.Size([4045, 48])
attention_mask  : torch.Size([4045, 48])
Alignment check : PASS



In [40]:
idx = 0

print("Image ID:")
print(train_tokenized["image_ids"][idx])

print("\nCaption index:")
print(train_tokenized["caption_indices"][idx])

print("\nCaption:")
print(train_tokenized["captions"][idx])

print("\nInput IDs:")
print(train_tokenized["input_ids"][idx])

print("\nAttention mask:")
print(train_tokenized["attention_mask"][idx])

Image ID:
3472540184_b0420b921a.jpg

Caption index:
0

Caption:
An older boy chases a laughing younger boy on the grass .

Input IDs:
tensor([ 2025,  4697,  2933,   442,  1386,   257, 14376,  7099,  2933,   319,
          262,  8701,   764, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256])

Attention mask:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [41]:
TOKENIZED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.save(
    train_tokenized,
    TOKENIZED_DIR / "train.pt",
)

torch.save(
    val_tokenized,
    TOKENIZED_DIR / "val.pt",
)

torch.save(
    test_tokenized,
    TOKENIZED_DIR / "test.pt",
)

print("Saved tokenized files to:")
print(TOKENIZED_DIR)

Saved tokenized files to:
d:\Zfs_Clip_Image\zfs-clip-image-captioning\notebook\data\flickr8k\tokenized
